# Lab 02 — Baseline Anomaly Detection

本 lab 接在 Lab 01 後面。\
Lab 01 把原始計數器整理成特徵並存成 `features.csv`；\
這裡用 rolling baseline 把「偏離正常」的時間點標記出來。

**你會學到**

| 方法 | 特性 |
|---|---|
| Z-score（rolling mean/std） | 計算簡單、對稱分布效果好；對離群值敏感 |
| Robust Z-score（rolling median/MAD） | 對離群值穩健；適合有長尾雜訊的網路資料 |
| P95 閾值 | 直覺好解釋，適合向非技術人員說明 |

這三種方法最終都輸出一個 0/1 flag——\
`1` = 這個時間點這個維度超出正常範圍，`0` = 正常。


In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/lab02_pipeline_position.svg"),
    find_project_root() / "labs/self-study" / "diagrams/lab02_pipeline_position.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/lab02_pipeline_position.svg")


## Lab 02 概覽

### 學習目標

1. 理解 z-score 的數學推導與直觀意義
2. 掌握 robust z-score（MAD）對離群值更穩健的原因
3. 用 Recall 評估偵測效果，理解 precision / recall 的業務含義
4. 做出「可接受的誤報率」這個業務決策

### 前置條件

Lab 01 完成（`outputs/self-study/features.csv` 存在）

## 設計主線：閾值是營運政策，不是數學常數

本章的實務連結是 on-call 負擔。Z-score、robust Z-score 與 P95 都能產生 anomaly flag，但 flag 太多會讓工程師忽略告警，flag 太少會漏掉事件。

設計偵測規則時請問三個問題：

1. **誤報預算**：一天可以接受幾次不需要處理的告警？不同 severity 是否該有不同 threshold？
2. **漏報成本**：流量尖峰、錯誤率、丟棄率、broadcast storm 的漏報代價是否相同？
3. **防抖機制**：單點異常是否要立刻告警，還是需要 deadband 或持續時間確認？

設計原則：threshold 要用營運成本校準。統計分數只提供候選訊號，不替你決定值班策略。


## 單一欄位異常判讀指南

在設定閾值之前，先理解每個欄位的異常意義。
同樣是「數值升高」，代表的問題可能完全不同。

| 欄位 / 特徵 | 升高代表 | 降低代表 | 你應該優先看哪個方向 |
|---|---|---|---|
| `traffic_total` | 備份、DDoS、業務高峰 | 連線中斷、設備下線 | **兩個方向都要** |
| `error_rate` | 線材、NIC、duplex mismatch | 問題解決（好事） | **只看升高** |
| `discard_rate` | Queue 滿、壅塞 | 問題解決 | **只看升高** |
| `broadcast_total` | ARP storm、switch loop | 正常（broadcast 本來就少） | **只看升高** |
| `avg_packet_size` | 大檔傳輸、備份 | 大量小封包（掃描？） | **兩個方向都要** |

**陷阱**：`error_rate` 在流量極低時（例如深夜 ucast ≈ 0）會出現統計假象。
這不是真實的線路問題，是分母趨近於 0 的數學現象。
Lab 01 的 `safe_div()` 處理了除以 0，但極低流量時的高 error_rate 仍需人工確認。



In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "environment.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "outputs" / "self-study"

features_path = DATA_PROCESSED / "features.csv"
if not features_path.exists():
    raise FileNotFoundError(
        f"找不到 {features_path}\n"
        "請先完整執行 Lab 01（包含 Step 6 的儲存步驟）。"
    )

df = pd.read_csv(features_path, parse_dates=["timestamp"])
events = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv",
                     parse_dates=["start_time", "end_time"])

print(f"載入特徵：{len(df):,} 筆  |  {df.shape[1]} 欄  |  "
      f"{df.port_id.nunique()} 個 port")


---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 8 分鐘，請先不要執行 cell。

---

## 📖 概念：Z-score — 偏離了幾個「正常波動單位」

### 核心問題

流量從 1,000 跳到 10,000——這個變化大嗎？
光看數字不夠，因為 10,000 對某些 port 是正常，對另一個 port 是災難。

Z-score 的解法：**用「這個 port 自己的正常波動幅度」當作尺**。

```
         Range of variation (schematic)

               ↑ Traffic
               │
  anomaly ── 3 │··············· ← z > 3: outside 99.7% of normal points
               │     ╭─╮
  normal  0~2  │ ───╯   ╰─── ← most time points fall here
               │
  anomaly ─ -3 │··············· ← z < -3: traffic drop (possible link down)
               └──────────────→ Time
```

### 閾值 3 的含義

| z-score | 直觀感受 | 統計意義（常態分布下）|
|---|---|---|
| 0 ~ 2 | 正常日常波動 | 95% 的Time都在這裡 |
| 2 ~ 3 | 略高，值得觀察 | 偶爾發生，不一定是事件 |
| > 3 | 明顯異常 | 每 333 個點才出現一次 |
| > 5 | 幾乎確定是事件 | 極罕見 |

**閾值 3 是統計慣例，不是魔法數字。**
你的資料可能不是常態分布，實際誤報率要靠 Lab 02 Step 3 的 Recall 評估來確認。

📎 視覺說明：[Standard score — Wikipedia](https://en.wikipedia.org/wiki/Standard_score)（有鐘形圖）

---

## 📖 概念：Robust Z-score — 抗離群值的版本

### Z-score 的弱點

```
normal sequence:  1  1  1  1  100  1  1  1
standard mean = 13.25  ← pulled up by 100
standard std  = 33.14  ← inflated by 100

result: subsequent "1" z-score = (1 - 13.25) / 33.14 = -0.37
→ clearly normal, yet z-score near 0 — the real anomaly cannot be detected
```

異常值污染了 baseline，讓整個偵測系統失靈。

### Robust Z-score 的解法：用中位數代替平均數

```
normal sequence:  1  1  1  1  100  1  1  1
median = 1   ← unaffected by 100
MAD  = 0     ← deviation of most points is near 0

intuition: "most ports carry traffic of 1 — how far does this value deviate?"
```

**中位數**（Median）和 **MAD**（Median Absolute Deviation）不受極端值影響，
適合有長尾雜訊的網路資料。

| 方法 | 用什麼代表「中心」| 用什麼代表「波動」| 對離群值的反應 |
|---|---|---|---|
| 標準 z-score | 平均數 | 標準差 | 敏感（被拉偏）|
| Robust z-score | 中位數 | MAD | 穩健（不受影響）|

📎 延伸閱讀：[Median absolute deviation — Wikipedia](https://en.wikipedia.org/wiki/Median_absolute_deviation)


## Step 1 — 定義監控維度與偵測方法

不是每個特徵都需要偵測「高」和「低」。\
例如 `error_rate` 升高才是問題（降低是好事），`traffic_total` 則兩個方向都要注意。

下面的 `WATCH` 字典定義：特徵名稱 → (rolling 欄位前綴, 偵測方向 `high` / `low` / `both`)


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

In [ ]:
# (特徵欄位, 說明, 偵測方向)
WATCH = [
    ("traffic_total",   "Traffic總量",       "both"),
    ("error_rate",      "封包錯誤率",     "high"),
    ("discard_rate",    "封包丟棄率",     "high"),
    ("broadcast_total", "broadcast 封包", "high"),
    ("avg_packet_size", "平均封包大小",   "both"),
]

Z_THRESH  = 3.0   # z-score threshold（可調整看效果）
RZ_THRESH = 4.0   # robust z-score threshold

def safe_div(a, b):
    return np.where(b > 0, a / b, 0.0)

result = df.copy()

for col, label, direction in WATCH:
    mean   = result[f"{col}__mean_1h"]
    std    = result[f"{col}__std_1h"].clip(lower=1e-9)
    median = result[f"{col}__median_1d"]
    mad    = result[f"{col}__mad_1d"].clip(lower=1e-9)
    p95    = result[f"{col}__p95_1d"]
    raw    = result[col]

    z  = safe_div(raw - mean,   std)
    rz = safe_div(raw - median, 1.4826 * mad)   # 1.4826 → 常態分布校正係數

    if direction in ("high", "both"):
        result[f"{col}__flag_z_high"]  = (z  >  Z_THRESH ).astype(int)
        result[f"{col}__flag_rz_high"] = (rz >  RZ_THRESH).astype(int)
        result[f"{col}__flag_p95"]     = (raw > p95       ).astype(int)
    if direction in ("low", "both"):
        result[f"{col}__flag_z_low"]   = (z  < -Z_THRESH ).astype(int)
        result[f"{col}__flag_rz_low"]  = (rz < -RZ_THRESH).astype(int)

flag_cols = [c for c in result.columns if "__flag_" in c]
print(f"產生 {len(flag_cols)} 個 anomaly flag 欄位：")
for c in flag_cols:
    rate = result[c].mean()
    print(f"  {c}: flag rate = {rate:.2%}")


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 2 — 三 panel 視覺化偵測結果

這張圖有三個 panel，請由上往下讀，不要只看紅點。

1. **上圖**：原始流量、rolling mean、mean ± 3σ 陰影帶
2. **中圖**：z-score 隨Time的變化；紅色虛線 = ±3σ 閾值
3. **下圖**：偵測到的 flag（點標記）

紅色陰影 = 已知事件時段。


In [ ]:
PORT = "port-id7427"
COL  = "traffic_total"

p = result[result["port_id"] == PORT].copy().sort_values("timestamp")
p_ev = events[events["port_id"] == PORT]

def shade_events(ax):
    for _, ev in p_ev.iterrows():
        ax.axvspan(ev.start_time, ev.end_time, alpha=0.13, color="crimson", zorder=0)

mean_s = p[f"{COL}__mean_1h"]
std_s  = p[f"{COL}__std_1h"].clip(lower=1e-9)
z_s    = (p[COL] - mean_s) / std_s

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1 — raw + baseline band
axes[0].plot(p["timestamp"], p[COL], color="lightgray", lw=0.8, label="raw")
axes[0].plot(p["timestamp"], mean_s, color="steelblue", lw=1.6, label="mean_1h")
axes[0].fill_between(p["timestamp"],
    mean_s - Z_THRESH * std_s, mean_s + Z_THRESH * std_s,
    alpha=0.15, color="steelblue", label=f"mean ± {Z_THRESH}σ")
axes[0].set_ylabel("Bytes / 5 min"); axes[0].legend(fontsize=9)
axes[0].set_title(f"{PORT} — {COL}: baseline anomaly detection (red shading = event)")

# Panel 2 — z-score time series
axes[1].plot(p["timestamp"], z_s, color="darkorange", lw=1, label="z-score")
axes[1].axhline( Z_THRESH, color="crimson",  ls="--", lw=1)
axes[1].axhline(-Z_THRESH, color="steelblue", ls="--", lw=1)
axes[1].set_ylabel("Z-score"); axes[1].legend(fontsize=9)

# Panel 3 — flag markers
flagged_high = p[p[f"{COL}__flag_z_high"] == 1]
flagged_low  = p[p[f"{COL}__flag_z_low"]  == 1] if f"{COL}__flag_z_low" in p.columns else pd.DataFrame()
axes[2].scatter(flagged_high["timestamp"], flagged_high[COL],
                color="crimson", s=12, zorder=3, label="flag_high")
if not flagged_low.empty:
    axes[2].scatter(flagged_low["timestamp"], flagged_low[COL],
                    color="steelblue", s=12, zorder=3, label="flag_low")
axes[2].plot(p["timestamp"], p[COL], color="lightgray", lw=0.6, alpha=0.7)
axes[2].set_ylabel("Bytes / 5 min"); axes[2].legend(fontsize=9)

for ax in axes:
    shade_events(ax)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3 — 用 Recall 評估偵測效果

Recall（召回率）= 真實事件中，有多少比例被偵測到。

```
Recall = detected event time points / total event time points
```

Recall = 1.0 → 完美偵測到所有事件時段。\
Recall = 0.0 → 完全沒偵測到。

**注意**：高 recall 但低 precision 代表誤報很多（假陽性多）。\
Lab 02 只看 recall；Lab 05 會處理 alert reduction（降低誤報）。


In [ ]:
# 用主要 flag 評估（z-score high for traffic_total）
EVAL_FLAG = "traffic_total__flag_z_high"

recall_rows = []
for _, ev in events.iterrows():
    mask = (
        (result["port_id"]   == ev["port_id"]) &
        (result["timestamp"] >= ev["start_time"]) &
        (result["timestamp"] <= ev["end_time"])
    )
    ev_points = result[mask]
    if ev_points.empty:
        continue
    detected = ev_points[EVAL_FLAG].sum()
    total    = len(ev_points)
    recall_rows.append({
        "port_id":    ev["port_id"],
        "event_type": ev["event_type"],
        "duration_h": round((ev["end_time"] - ev["start_time"]).total_seconds() / 3600, 1),
        "total_pts":  total,
        "detected":   detected,
        "recall":     round(detected / total, 3) if total else 0.0,
    })

recall_df = pd.DataFrame(recall_rows)
display(recall_df.sort_values("recall", ascending=False))
print(f"\n平均 recall（traffic_total，z-score）：{recall_df.recall.mean():.2%}")


---

## ⚖️ 方法比較：Z-score vs Robust Z-score vs P95 百分位

| 方法 | 核心統計量 | 優點 | 缺點 | 最適合 |
|---|---|---|---|---|
| **Z-score** | mean / std | 計算快、有理論基礎、對稱分布精確 | 對離群值敏感；流量峰值拉偏 baseline | 流量分布接近常態、雜訊少的環境 |
| **Robust Z-score** | median / MAD | 不被離群值拉偏；長尾資料穩健 | 對快速變化的趨勢反應較慢 | 有周期性大峰值（備份、批次作業）的網路 |
| **P95 百分位** | 第 95 百分位數 | 直觀、易向非技術人員說明 | 固定百分位意味著永遠有 5% 被標記 | 向業務層說明時，或作為 SLO 閾值 |

### 本課程同時使用 Z-score and Robust Z-score 的原因

網路流量的分布通常是**右偏長尾**：大多數Time流量低而穩定，
偶發高峰（備份、業務尖峰）把 mean 和 std 拉高。在這種情況下，
普通 z-score 的 baseline 被高峰拉偏後，正常低流量時段反而容易出現負向假陽性。

Robust z-score 用 median 作為 baseline，高峰不影響它的中心估計。
兩種方法同時使用，可以互補：若只有一種方法觸發，代表信號較弱；兩種都觸發，
可信度更高。

### 如何選擇？

```
Is your data distribution approximately normal (stable traffic)?
    → Z-score is sufficient

Do you have regular high-traffic periods (nightly backups, batch jobs)?
    → Robust Z-score or P95

Do you need to explain the threshold to non-technical stakeholders?
    → P95 ("alert when traffic exceeds 95% of the past day's values")
```

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4 — 跨 port 比較：Broadcast Storm

Broadcast storm 的特性通常不是單一 port 獨自變化，而是多個 port 同時升高。\
這張圖把所有 port 的 `broadcast_total` 疊在一起，觀察同步升高的時間點。


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

port_colors = {
    "port-id7427": "steelblue", "port-id7428": "darkorange",
    "port-id7429": "seagreen",  "port-id7430": "mediumpurple",
    "port-id7431": "crimson",
}

for port_id, color in port_colors.items():
    p = result[result["port_id"] == port_id].sort_values("timestamp")
    axes[0].plot(p["timestamp"], p["broadcast_total"],
                 color=color, lw=1, alpha=0.8, label=port_id)

axes[0].set_ylabel("broadcast_total"); axes[0].set_title("All ports: broadcast_total comparison")
axes[0].legend(fontsize=8, ncol=3)

# Panel 2 — broadcast flag (z-score) by port
for port_id, color in port_colors.items():
    p = result[result["port_id"] == port_id].sort_values("timestamp")
    flagged = p[p["broadcast_total__flag_z_high"] == 1]
    axes[1].scatter(flagged["timestamp"], [port_id]*len(flagged),
                    color=color, s=10, alpha=0.7)

axes[1].set_ylabel("Port"); axes[1].set_title("Broadcast storm flag（z-score high）")

# Broadcast storm 事件陰影
for _, ev in events[events["event_type"] == "broadcast_storm"].iterrows():
    for ax in axes:
        ax.axvspan(ev.start_time, ev.end_time, alpha=0.12, color="gold", zorder=0)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 5 — 儲存 anomaly flags

把所有 flag 欄位和原始特徵一起存檔，供 Lab 05（Alert Reduction）使用。


In [ ]:
out_path = DATA_PROCESSED / "baseline_anomaly_flags.csv"
result.to_csv(out_path, index=False)
print(f"✅ 儲存完成：{out_path}")
print(f"   筆數：{len(result):,}  |  Flag 欄位：{len(flag_cols)}")


---
💬 **討論暫停 ▸ 全班討論** — 停止執行，分享你的觀察。

---

## ⚠ 人類驗證點 #2 — 閾值是你的業務決策，不是統計結論

Step 1 的 `Z_THRESH = 3.0` 和 `RZ_THRESH = 4.0` 是起點，不是答案。

**問題不是「哪個閾值讓 recall 最高」——而是「這個 recall / precision 組合，我們能接受嗎？」**

| 場景 | 偏向降低閾值（更敏感） | 偏向升高閾值（更保守） |
|---|---|---|
| 金融機構、關鍵基礎設施 | 寧可多告警，不能漏掉真實事件 | — |
| 一般辦公室網路 | — | 運維人員已疲於應付告警，誤報成本高 |
| 自動化回應（自動封 port） | 非常保守，誤報代價極高 | 絕對需要人工確認 |

**在繼續之前：**

1. 看 Step 3 的 recall 表。哪種事件的 recall 最低？這是可以接受的嗎？
2. `large_file_backup` 的 recall 可能很低——為什麼？\
   （提示：它讓 `traffic_total` 升高，但 `avg_packet_size` 也升高。z-score 只看一個維度。）
3. 如果你把 `Z_THRESH` 降到 2.0，誤報率會怎樣變化？這對運維團隊的工作量有什麼影響？

**AI 給你一個分數；你決定這個分數的切點。沒有客觀正確的閾值。**


---
🔧 **探索練習** — 修改指定參數，觀察結果變化。

---

## 🔧 探索練習：閾值的敏感度

**任務**：在 Step 1 的 code cell 找到以下變數，分別修改並重新執行 Step 2 和 Step 3。

```python
Z_THRESH = 3.0   # change to 2.5 and observe how flag rate and recall shift
RZ_THRESH = 4.0  # change to 3.0 and observe how robust flag changes
```

**如何判斷結果是否合理？**

- Recall（召回率）提升是好事，但代價是什麼？查看 Step 2 圖的紅點密度。
- 如果「正常時段」出現大量紅點，表示誤報率偏高——運維人員會忽略告警。
- 如果 Recall 仍低於 0.7（70%），表示真實事件漏報率過高——代表閾值太保守。

**讓你重新考慮的信號**

- 每小時的 flag rate 超過 20%（每 5 分鐘就有一個點被標記）→ 閾值可能太低
- 已知的 `error_burst` 事件完全未被偵測到 → 可能需要針對 `error_rate` 降低閾值

---

> 「Recall = 1.0 是完美的嗎？代價是什麼？」

> 「為什麼 `large_file_backup` 事件的 recall 通常比 `error_burst` 低？（提示：哪個特徵最能代表備份行為？）」

> 「如果你的團隊目前每天收到 500 則告警，降低閾值到 2.5σ 之後會發生什麼？這個改變需要誰批准？」

## 練習與檢查點

1. 把 `Z_THRESH` 從 `3.0` 改成 `2.5`，重新執行 Step 2。\
   Flag rate 有什麼變化？Recall 有提升嗎？代價是什麼？

2. 把 `PORT` 改成 `port-id7431`，Step 2 的三張圖哪個事件最明顯？

3. Robust Z-score 用的是 median 和 MAD 而不是 mean 和 std。\
   在什麼情況下 robust z-score 的表現會明顯優於普通 z-score？

---

**下一步 → Lab 03 — SPC for Anomaly Detection**：\
使用統計製程控制（Statistical Process Control）方法，\
在不假設資料為常態分布的條件下偵測異常。
